
[notice] A new release of pip is available: 23.2.1 -> 24.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

2024-06-25 18:57:37.197308: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-06-25 18:57:43.338409: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


2024-06-25 18:57:52.461186: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:2b:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-06-25 18:57:52.601329: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:2b:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-06-25 18:57:52.601374: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:2b:00.0/numa_node
Your kernel may have been built without NUMA support.


In [4]:
import cv2
import numpy as np
import os
%matplotlib nbagg
import mediapipe as mp

In [5]:
mp_holistic = mp.solutions.holistic # Holistic model
mp_drawing = mp.solutions.drawing_utils # Drawing utilities

In [6]:
def mediapipe_detection(image, model):
    image =  cv2.cvtColor(image, cv2.COLOR_BGR2RGB)# COLOR CONVERSION BGR 2 RGB
    image.flags.writeable = False                  # Image is no longer writeable
    results = model.process(image)                 # Make prediction
    image.flags.writeable = True                   # Image is now writeable
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR) # COLOR COVERSION RGB 2 BGR
    return image, results

In [7]:
def draw_landmarks(image,results):
   
    mp_drawing.draw_landmarks(image,results.pose_landmarks,mp_holistic.POSE_CONNECTIONS)
    mp_drawing.draw_landmarks(image,results.left_hand_landmarks,mp_holistic.HAND_CONNECTIONS)
    mp_drawing.draw_landmarks(image,results.right_hand_landmarks,mp_holistic.HAND_CONNECTIONS)

In [8]:
def draw_styled_landmarks(image,results):
   
    mp_drawing.draw_landmarks(image,results.pose_landmarks,mp_holistic.POSE_CONNECTIONS,
                              mp_drawing.DrawingSpec(color=(80,22,10),thickness=2,circle_radius=4),
                              mp_drawing.DrawingSpec(color=(80,44,121),thickness=2,circle_radius=2))
    mp_drawing.draw_landmarks(image,results.left_hand_landmarks,mp_holistic.HAND_CONNECTIONS,
                              mp_drawing.DrawingSpec(color=(121,22,76),thickness=2,circle_radius=4),
                              mp_drawing.DrawingSpec(color=(121,44,250),thickness=2,circle_radius=1))
    mp_drawing.draw_landmarks(image,results.right_hand_landmarks,mp_holistic.HAND_CONNECTIONS,
                              mp_drawing.DrawingSpec(color=(245,117,66),thickness=2,circle_radius=4),
                              mp_drawing.DrawingSpec(color=(245,66,230),thickness=2,circle_radius=2))

In [9]:
def extract_keypoints(results):
    pose = np.array([[res.x, res.y, res.z, res.visibility] for res in results.pose_landmarks.landmark]).flatten() if results.pose_landmarks else np.zeros(33*4)

    lh = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(21*3)
    rh = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(21*3)
    return np.concatenate([pose, lh, rh])

In [10]:
DATA_PATH = os.path.join('BL_Data')
actions = np.array(['Ajebaje', 'Akash', 'Alada', 'Allah', 'Asha', 'Bakko', 'Bank',
         'Bari', 'Bebsha', 'Bepar', 'Beyam', 'Bhromon', 'Bibaho', 'Biggan',
         'Biruddhe', 'Bisoy', 'Boi', 'Boka', 'Brisiti', 'Camera', 'Cha',
         'Chawa', 'Churanto', 'Dam', 'Daraw', 'Dawat', 'Dharona', 'Dhowa',
         'Dokander', 'Dol', 'Dowa_kora', 'Druto', 'Dupur', 'Durgondho',
         'Ful', 'Gari', 'Ghi', 'Ghori', 'Ghosito_howa', 'Ghumano', 'Hat',
         'Hasi', 'Hassokor', 'Injection', 'Jailkhana', 'Jinish', 'Jogajog',
         'Kachi', 'Kapor', 'Kashi', 'Khawa', 'Khoma', 'Klanto', 'Kukur',
         'Mach', 'Matha', 'Matha_betha', 'Mongol', 'Moyla', 'Name',
         'Norachora', 'Ojon', 'Onushoron', 'Opomanjonok', 'Oshustho',
         'Oishodh', 'Petuk', 'Phone', 'Pochondo', 'Porikkha', 'Poriskar',
         'Prostut', 'Protarona', 'Raat', 'Rajdhani', 'Rasta', 'Sabdhan',
         'Shajano', 'Shasti', 'Shokti', 'Sorto', 'Shotru', 'Shokal', 'Soman',
         'Somossha', 'Somoy', 'Songbad', 'Sonkirno', 'Shosta', 'Table',
         'Taka', 'Tamasha', 'Tapmatra', 'Tarikh', 'Toiri_kora', 'Tumi',
         'Unnoto', 'Upor', 'Vaggo', 'Bhalo', 'Vari', 'Vule_jawa'])


no_sequences = 30
sequence_length = 30

In [11]:
actions

array(['Ajebaje', 'Akash', 'Alada', 'Allah', 'Asha', 'Bakko', 'Bank',
       'Bari', 'Bebsha', 'Bepar', 'Beyam', 'Bhromon', 'Bibaho', 'Biggan',
       'Biruddhe', 'Bisoy', 'Boi', 'Boka', 'Brisiti', 'Camera', 'Cha',
       'Chawa', 'Churanto', 'Dam', 'Daraw', 'Dawat', 'Dharona', 'Dhowa',
       'Dokander', 'Dol', 'Dowa_kora', 'Druto', 'Dupur', 'Durgondho',
       'Ful', 'Gari', 'Ghi', 'Ghori', 'Ghosito_howa', 'Ghumano', 'Hat',
       'Hasi', 'Hassokor', 'Injection', 'Jailkhana', 'Jinish', 'Jogajog',
       'Kachi', 'Kapor', 'Kashi', 'Khawa', 'Khoma', 'Klanto', 'Kukur',
       'Mach', 'Matha', 'Matha_betha', 'Mongol', 'Moyla', 'Name',
       'Norachora', 'Ojon', 'Onushoron', 'Opomanjonok', 'Oshustho',
       'Oishodh', 'Petuk', 'Phone', 'Pochondo', 'Porikkha', 'Poriskar',
       'Prostut', 'Protarona', 'Raat', 'Rajdhani', 'Rasta', 'Sabdhan',
       'Shajano', 'Shasti', 'Shokti', 'Sorto', 'Shotru', 'Shokal',
       'Soman', 'Somossha', 'Somoy', 'Songbad', 'Sonkirno', 'Shosta',
       'Tab

In [12]:
for action in actions:
    for sequence in range(no_sequences):
        try:
            os.makedirs(os.path.join(DATA_PATH,action,str(sequence)))
        except:
            pass


In [11]:
import os

# NEW Read images from folder
image_folder = 'FInal_Dataset'
image_files = [os.path.join(image_folder, f) for f in os.listdir(image_folder) if f.endswith('.jpg')]
x=0
# Set mediapipe model
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:

    # Loop through actions
    for action in actions:
        # Loop through sequences aka videos
        for sequence in range(no_sequences):
            # Loop through video length aka sequence length
            for frame_num in range(sequence_length):

                # Read feed
                image = cv2.imread(image_files[sequence+x])

                # Make detections
                image, results = mediapipe_detection(image, holistic)

                # Draw landmarks
                draw_styled_landmarks(image, results)

                # Apply wait logic
                if frame_num == 0:
                    cv2.putText(image, 'STARTING COLLECTION', (120,200),
                               cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255, 0), 4, cv2.LINE_AA)
                    cv2.putText(image, 'Collecting frames for {} Video Number {}'.format(action, sequence), (15,12),
                               cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1, cv2.LINE_AA)
                    # Show to screen
                     # Resize image to fit screen
                    cv2.imshow('OpenCV Feed', image)
                    cv2.waitKey(100)
                else:
                    cv2.putText(image, 'Collecting frames for {} Video Number {}'.format(action, sequence), (15,12),
                               cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1, cv2.LINE_AA)
                    # Show to screen
                      # Resize image to fit screen
                    cv2.imshow('OpenCV Feed', image)

                # Export keypoints
                keypoints = extract_keypoints(results)
                npy_path = os.path.join(DATA_PATH, action, str(sequence), str(frame_num))
                np.save(npy_path, keypoints)
                # Break gracefully
                if cv2.waitKey(10) & 0xFF == ord('q'):
                    break

        x+=30
    cv2.destroyAllWindows()

D:\BDSLP-Transfomer\venv\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


In [12]:
cv2.destroyAllWindows()

In [80]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

In [81]:
label_map = {label:num for num, label in enumerate(actions)}

In [82]:
label_map

{'Ajebaje': 0,
 'Akash': 1,
 'Alada': 2,
 'Allah': 3,
 'Asha': 4,
 'Bakko': 5,
 'Bank': 6,
 'Bari': 7,
 'Bebsha': 8,
 'Bepar': 9,
 'Beyam': 10,
 'Bhromon': 11,
 'Bibaho': 12,
 'Biggan': 13,
 'Biruddhe': 14,
 'Bisoy': 15,
 'Boi': 16,
 'Boka': 17,
 'Brisiti': 18,
 'Camera': 19,
 'Cha': 20,
 'Chawa': 21,
 'Churanto': 22,
 'Dam': 23,
 'Daraw': 24,
 'Dawat': 25,
 'Dharona': 26,
 'Dhowa': 27,
 'Dokander': 28,
 'Dol': 29,
 'Dowa_kora': 30,
 'Druto': 31,
 'Dupur': 32,
 'Durgondho': 33,
 'Ful': 34,
 'Gari': 35,
 'Ghi': 36,
 'Ghori': 37,
 'Ghosito_howa': 38,
 'Ghumano': 39,
 'Hat': 40,
 'Hasi': 41,
 'Hassokor': 42,
 'Injection': 43,
 'Jailkhana': 44,
 'Jinish': 45,
 'Jogajog': 46,
 'Kachi': 47,
 'Kapor': 48,
 'Kashi': 49,
 'Khawa': 50,
 'Khoma': 51,
 'Klanto': 52,
 'Kukur': 53,
 'Mach': 54,
 'Matha': 55,
 'Matha_betha': 56,
 'Mongol': 57,
 'Moyla': 58,
 'Name': 59,
 'Norachora': 60,
 'Ojon': 61,
 'Onushoron': 62,
 'Opomanjonok': 63,
 'Oshustho': 64,
 'Oishodh': 65,
 'Petuk': 66,
 'Phone': 67,
 '

In [83]:
sequences, labels = [], []
for action in actions:
    for sequence in range(no_sequences):
        window = []
        for frame_num in range(sequence_length):
            res = np.load(os.path.join(DATA_PATH, action, str(sequence), "{}.npy".format(frame_num)))
            window.append(res)
        sequences.append(window)
        labels.append(label_map[action])

In [84]:
np.array(sequences)

array([[[ 0.44661236,  0.10005587, -0.77693492, ...,  0.33768579,
          0.45896715, -0.05536904],
        [ 0.44706437,  0.09431598, -0.71271139, ...,  0.31364641,
          0.46158081, -0.04958243],
        [ 0.44802406,  0.09179849, -0.6983881 , ...,  0.32988217,
          0.46031386, -0.05987792],
        ...,
        [ 0.44956109,  0.0826077 , -0.76009506, ...,  0.31437683,
          0.46328658, -0.0480159 ],
        [ 0.44956818,  0.08253908, -0.75742292, ...,  0.342935  ,
          0.44961244, -0.05987213],
        [ 0.44957465,  0.08239426, -0.75014555, ...,  0.31531397,
          0.46258634, -0.04824198]],

       [[ 0.42362636,  0.08671962, -0.65002525, ...,  0.46281552,
          0.10629856, -0.01300193],
        [ 0.41726518,  0.08663775, -0.87809521, ...,  0.46873206,
          0.10028438, -0.01671414],
        [ 0.41969416,  0.08647107, -0.83414066, ...,  0.46734482,
          0.10152252, -0.01463428],
        ...,
        [ 0.42730114,  0.09091934, -0.85498035, ...,  

In [88]:
train = np.array(sequences)

In [89]:
train.shape

(3060, 30, 258)

In [90]:
test = to_categorical(labels).astype(int)

In [91]:
test.shape

(3060, 102)

In [92]:
train_data,test_data,train_labels, test_labels = train_test_split(train,test,test_size=0.40)

In [93]:
np.array(train_data),np.array(test_data),np.array(train_labels),np.array(test_labels)

(array([[[ 0.47872746,  0.3416599 , -1.14841676, ...,  0.        ,
           0.        ,  0.        ],
         [ 0.46205395,  0.34113848, -0.86384624, ...,  0.        ,
           0.        ,  0.        ],
         [ 0.46139213,  0.34088081, -0.77805823, ...,  0.        ,
           0.        ,  0.        ],
         ...,
         [ 0.44663742,  0.34802395, -0.81800288, ...,  0.        ,
           0.        ,  0.        ],
         [ 0.44651288,  0.34823745, -0.81987202, ...,  0.        ,
           0.        ,  0.        ],
         [ 0.44641528,  0.34847397, -0.82134652, ...,  0.        ,
           0.        ,  0.        ]],
 
        [[ 0.51884186,  0.15141191, -1.06945705, ...,  0.45040742,
           0.44308832, -0.0621889 ],
         [ 0.51203603,  0.15236844, -1.06561208, ...,  0.44741198,
           0.44387078, -0.05167269],
         [ 0.50967008,  0.15309407, -1.0874809 , ...,  0.44840181,
           0.44435582, -0.05229922],
         ...,
         [ 0.50746608,  0.1554516

In [94]:
train_data.shape

(1836, 30, 258)

In [96]:
test_data.shape

(1224, 30, 258)

In [97]:
import os
import keras
from keras import layers
from keras.applications.densenet import DenseNet121
import tensorflow as tf
from tensorflow_docs.vis import embed
from sklearn.metrics import precision_recall_curve, precision_score, recall_score, f1_score,accuracy_score,precision_recall_curve, classification_report
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import imageio
import cv2

In [116]:
class PositionalEmbedding(layers.Layer):
    def __init__(self, sequence_length, output_dim, **kwargs):
        super().__init__(**kwargs)
        self.position_embeddings = layers.Embedding(
            input_dim=sequence_length, output_dim=output_dim
        )
        self.sequence_length = sequence_length
        self.output_dim = output_dim

    def call(self, inputs):
        # The inputs are of shape: `(batch_size, frames, num_features)`
        inputs = tf.cast(inputs, self.compute_dtype)
        length = tf.shape(inputs)[1]
        positions = tf.range(start=0, limit=length, delta=1)
        embedded_positions = self.position_embeddings(positions)
        embedded_positions = tf.expand_dims(embedded_positions, axis=0)
        return inputs + embedded_positions

In [117]:
class TransformerEncoder(layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.dense_dim = dense_dim
        self.num_heads = num_heads
        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim, dropout=0.3
        )
        self.dense_proj = keras.Sequential(
            [
                layers.Dense(dense_dim, activation=keras.activations.gelu),
                layers.Dense(embed_dim),
            ]
        )
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()

    def call(self, inputs, mask=None):
        attention_output = self.attention(inputs, inputs, attention_mask=mask)
        proj_input = self.layernorm_1(inputs + attention_output)
        proj_output = self.dense_proj(proj_input)
        return self.layernorm_2(proj_input + proj_output)

In [118]:
def get_compiled_model(shape):
    sequence_length = shape[0]  # 30
    embed_dim = shape[1]        # 258
    dense_dim = 4
    num_heads = 1
    classes = 102

    inputs = keras.Input(shape=shape)
    x = PositionalEmbedding(
        sequence_length, embed_dim, name="frame_position_embedding"
    )(inputs)
    x = TransformerEncoder(embed_dim, dense_dim, num_heads, name="transformer_layer")(x)
    x = layers.GlobalMaxPooling1D()(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(classes, activation="softmax")(x)
    model = keras.Model(inputs, outputs)

    model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

def run_experiment(train_data, train_labels, test_data, test_labels):
    filepath = "video_classifier.weights.h5"
    checkpoint = keras.callbacks.ModelCheckpoint(
        filepath, save_weights_only=True, save_best_only=True, verbose=1
    )

    model = get_compiled_model(train_data.shape[1:])
    history = model.fit(
        train_data,
        train_labels,
        validation_split=0.15,
        epochs=1000,
        callbacks=[checkpoint],
    )

    model.load_weights(filepath)
    test_preds = model.predict(test_data)
    test_preds_classes = test_preds.argmax(axis=1)
    
    _, accuracy = model.evaluate(test_data, test_labels)
    print(f"Test accuracy: {round(accuracy * 100, 2)}%")

    print("Classification Report:")
    print(classification_report(test_labels.argmax(axis=1), test_preds_classes, target_names=[str(i) for i in range(102)]))

    # Plot Precision-Recall curve for each class
    plt.figure(figsize=(12, 8))
    for i in range(102):
        precision, recall, _ = precision_recall_curve(test_labels[:, i], test_preds[:, i])
        plt.plot(recall, precision, marker='.', label=f'Class {i}')

    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-Recall Curve for All Classes')
    plt.legend()
    plt.savefig('precision_recall_curve_all_classes.png')  # Save the figure
    plt.show()

    return model, test_preds_classes

In [119]:
trained_model,test_preds_classes = run_experiment(train_data, train_labels, test_data, test_labels)

Epoch 1/1000
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.0062 - loss: 5.6843 
Epoch 1: val_loss improved from inf to 4.32537, saving model to video_classifier.weights.h5
49/49 ━━━━━━━━━━━━━━━━━━━━ 7s 63ms/step - accuracy: 0.0064 - loss: 5.6785 - val_accuracy: 0.0725 - val_loss: 4.3254
Epoch 2/1000
44/49 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.0422 - loss: 4.7063
Epoch 2: val_loss improved from 4.32537 to 3.90664, saving model to video_classifier.weights.h5
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.0428 - loss: 4.6960 - val_accuracy: 0.0797 - val_loss: 3.9066
Epoch 3/1000
44/49 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.0865 - loss: 4.1131
Epoch 3: val_loss improved from 3.90664 to 3.21660, saving model to video_classifier.weights.h5
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.0871 - loss: 4.1016 - val_accuracy: 0.1848 - val_loss: 3.2166
Epoch 4/1000
38/49 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.1546 - loss: 3.4415
Epoch 4: val_loss improv

<IPython.core.display.Javascript object>

In [120]:
def check_actions(test_labels, test_preds_classes, actions, index):
    true_action = actions[np.argmax(test_labels[index])]
    predicted_action = actions[test_preds_classes[index]]
    print(f"True action: {true_action}")
    print(f"Predicted action: {predicted_action}")

In [129]:
res = trained_model.predict(test_data)

39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 


In [137]:
actions[np.argmax(res[4])]

'Unnoto'

In [138]:
actions[test_preds_classes[4]]

'Unnoto'

In [121]:
check_actions(test_labels, test_preds_classes, actions, index=0)

True action: Biggan
Predicted action: Biggan


In [115]:
trained_model.summary()

Model: "functional_19"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_18 (InputLayer)     │ (None, 30, 258)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ frame_position_embedding        │ (None, 30, 258)        │         7,740 │
│ (PositionalEmbedding)           │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_layer               │ (None, 30, 258)        │       270,646 │
│ (TransformerEncoder)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_9          │ (None, 258)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_19 (Dropout)            │ (None, 258)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_29 (Dense)                │ (None, 102)            │        26,418 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 914,414 (3.49 MB)

 Trainable params: 304,804 (1.16 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 609,610 (2.33 MB)

In [126]:
yhat_probs = trained_model.predict(test_data)

# Check if y_train is one-hot encoded or just class labels
  # One-hot encoded
ytrue = np.argmax(test_labels, axis=1)


# Predict class labels from probabilities
yhat = np.argmax(yhat_probs, axis=1)

# Calculate precision, recall, and F1-score for each class
precision_scores = precision_score(ytrue, yhat, average=None)
recall_scores = recall_score(ytrue, yhat, average=None)
f1_scores = f1_score(ytrue, yhat, average=None)

# Calculate macro-averaged precision, recall, and F1-score
macro_precision = np.mean(precision_scores)
macro_recall = np.mean(recall_scores)
macro_f1 = np.mean(f1_scores)

print("Macro-Averaged Precision:", macro_precision)
print("Macro-Averaged Recall:", macro_recall)
print("Macro-Averaged F1-score:", macro_f1)

# Calculate precision-recall curve
 # Multi-class classification
    # Choose the class for which to plot the Precision-Recall curve
class_index = 1  # Adjust as needed
precision, recall, thresholds = precision_recall_curve(ytrue == class_index, yhat_probs[:, class_index])


# Plot precision-recall curve
plt.figure()
plt.plot(recall, precision, marker='.')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')

# Save the plot as an image file
plt.savefig('fprecision_recall_curve.png')

# Show the plot
plt.show()

39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
Macro-Averaged Precision: 0.9534254160551737
Macro-Averaged Recall: 0.9550195175195176
Macro-Averaged F1-score: 0.9500113361200268


<IPython.core.display.Javascript object>

In [127]:
print(f"Accuracy: {accuracy_score(ytrue, yhat) * 100} %")

Accuracy: 95.1797385620915 %
